## This notebook analyses growth of startsup's in India during 2016 to 2022.

# Import Libraries

In [1]:
%load_ext cudf.pandas
import numpy as np 
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# from wordcloud import WordCloud
import string

# Visualisations

In [2]:
df = pd.read_csv('/home/dias-benchmarks/notebooks/sandhyakrishnan02/indian-startup-growth-analysis/input/indian-startup-recognized-by-dpiit/Startup_Counts_Across_India.csv')
df.head().style.background_gradient(cmap='coolwarm')

,S No.,Year,State,Industry,Count
0,1,2022,Andaman and Nicobar Islands,Agriculture,1
1,2,2022,Andaman and Nicobar Islands,AR VR (Augmented + Virtual Reality),1
2,3,2022,Andaman and Nicobar Islands,Construction,1
3,4,2022,Andaman and Nicobar Islands,Internet of Things,1
4,5,2022,Andaman and Nicobar Islands,Marketing,1


# -- STEFANOS -- Replicate Data

In [3]:
# factor = 2500
factor = 1000
df = pd.concat([df]*factor, ignore_index=True)
df.info()


<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 5891000 entries, 0 to 5890999
Data columns (total 5 columns):
 #   Column    Dtype
---  ------    -----
 0   S No.     int64
 1   Year      int64
 2   State     object
 3   Industry  object
 4   Count     int64
dtypes: int64(3), object(2)
memory usage: 319.8+ MB


In [4]:
memory_gb = df['Industry'].memory_usage(deep=True) / (1024 ** 3)
memory_gb

0.10708533599972725

## Industry Vs Year

In [4]:
# STEFANOS: Disable plotting
# fig = px.scatter(df,x="Industry", y="Year", size="Count", color="Count",template='plotly_dark', title="Industry Vs Year")
# fig.show()

## Year wise Startup Growth

In [5]:
# fig, ax = plt.subplots(1,1, figsize=(15, 6))
df_year = df['Year'].value_counts().sort_index()
# STEFANOS: Disable plotting
# ax.bar(df_year.index, df_year, width=0.55,linewidth=0.7, color = 'purple')
# for i in df_year.index:
#     ax.annotate(f"{df_year[i]}",xy=(i, df_year[i] + 100),
#                    va = 'center', ha='center')
# ax.set_ylim(0, 1300)    
# fig.text(0.1, 0.95, "Growth of Startup's from 2016-2022", fontsize=15, fontweight='bold')    

# ax.grid(axis='y', linestyle='-', alpha=0.4)  

## Various Industries and Its Count

In [6]:
%%time
#cell 3
df.groupby('Industry').size().sort_values(ascending=False).to_frame().style.background_gradient(cmap='coolwarm')

CPU times: user 43.9 ms, sys: 7.97 ms, total: 51.9 ms
Wall time: 48.6 ms


,0
Industry,
IT Services,1760000
Education,1590000
Construction,1580000
Healthcare & Lifesciences,1570000
Food & Beverages,1560000
Professional & Commercial Services,1560000
Agriculture,1520000
Travel & Tourism,1470000
Retail,1440000


In [8]:
%%cudf.pandas.profile
#cell 3
df.groupby('Industry').size().sort_values(ascending=False).to_frame().style.background_gradient(cmap='coolwarm')

,0
Industry,
IT Services,1760000
Education,1590000
Construction,1580000
Healthcare & Lifesciences,1570000
Food & Beverages,1560000
Professional & Commercial Services,1560000
Agriculture,1520000
Travel & Tourism,1470000
Retail,1440000


                                                                                                                
                                           Total time elapsed: 0.453 seconds                                    
                                         4 GPU function calls in 0.062 seconds                                  
                                         4 CPU function calls in 0.087 seconds                                  
                                                                                                                
                                                         Stats                                                  
                                                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                   ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.groupby          │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ GroupBy.size               │ 1          │ 0.053       │ 0.053       │ 0          │ 0.000       │ 0.000       │
│ Series.sort_values         │ 1          │ 0.007       │ 0.007       │ 0          │ 0.000       │ 0.000       │
│ Series.to_frame            │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ Styler.background_gradient │ 0          │ 0.000       │ 0.000       │ 1          │ 0.002       │ 0.002       │
│ object.__repr__            │ 0          │ 0.000       │ 0.000       │ 1          │ 0.001       │ 0.001       │
│ Styler._repr_html_         │ 0          │ 0.000       │ 0.000       │ 1          │ 0.083       │ 0.083       │
│ Styler._repr_latex_        │ 0          │ 0.000       │ 0.000       │ 1          │ 0.001       │ 0.001       │
└────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- Styler.background_gradient
- object.__repr__
- Styler._repr_html_
- Styler._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=764030;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [14]:
memory_gb = df['Industry'].memory_usage(deep=True) / (1024 ** 3)
memory_gb

1.0708533264696598

## Top 20 Startup Industries from 2016

In [7]:
# fig, ax = plt.subplots(1,1, figsize=(20, 6))
df_ind = df['Industry'].value_counts().iloc[:20]

# STEFANOS: Disable plotting
# ax.bar(df_ind.index, df_ind, width=0.55,linewidth=0.7, color = 'pink')
# for i in df_ind.index:
#     ax.annotate(f"{df_ind[i]}",xy=(i, df_ind[i] + 20),va = 'center', ha='center')
# ax.set_ylim(0, 220)    
# fig.text(0.1, 0.95, "Top 20 Startup Industries from 2016", fontsize=15, fontweight='bold')    
# plt.xticks(rotation=90)
# ax.grid(axis='y', linestyle='-', alpha=0.4)  

## Top 20 States with Max Startup's

In [8]:
# fig, ax = plt.subplots(1,1, figsize=(20, 6))
X = df['State'].value_counts().iloc[:20]

# STEFANOS: Disable plotting
# ax.bar(X.index, X, width=0.55,linewidth=0.7, color = 'lightblue')
# for i in X.index:
#     ax.annotate(f"{X[i]}",xy=(i, X[i] + 50),
#                    va = 'center', ha='center')
# ax.set_ylim(0, 400)    
# fig.text(0.1, 0.95, "Top 20 States Having maximum Start'ups from 2016", fontsize=15, fontweight='bold')    
# plt.xticks(rotation=90)
# ax.grid(axis='y', linestyle='-', alpha=0.4)  

## AI related Startup's from 2016-2022

In [9]:
ds_list=['Internet of Things','AI','Robotics','Analytics','Computer Vision']
ds_df = df[df['Industry'].isin(ds_list)]

CPU times: user 337 ms, sys: 32.5 ms, total: 369 ms
Wall time: 369 ms


In [10]:
# STEFANOS: Disable plotting
# sns.set_style('whitegrid')
# sns.catplot(x='Year', hue = 'Industry', kind='count', data=ds_df,palette="Set3", height=8.27, aspect=11.7/8.27);
# plt.show()